# DocuDevs: Classify Products into a Category Tree

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/docudevs/python-examples/blob/main/10-product-classification.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/docudevs/python-examples)

File new grocery products into a category tree. A records case holds labelled products as evidence, the classifier files held-out products and flags the uncertain ones, a reviewer's correction becomes a memo instruction, and a holdout evaluation measures whether that instruction helps.

**What you'll learn:**
- Create a records case and import labelled records
- Upload and activate a category tree
- Classify products, wait for the job and read accuracy per tree level
- Work with flagged rows and review results
- Turn a correction into an approved memo instruction
- Measure the instruction with a holdout evaluation

**Requirements:** a DocuDevs API key and `docu-devs-api-client>=1.13.0`, the release with the case-bound records API and block uploads used here.

## The Use Case

A retailer keeps a category tree for its range: *Dairy > Cheese > Hard cheese*, *Beverages > Soft drinks > Carbonated drinks*, and so on. New products arrive from suppliers without a category, and each one has to be filed into the tree. The retailer already has thousands of products with a category, and those labels show how the team files things.

DocuDevs turns the labelled products into a **records** knowledge base. For a new product, the classifier looks at the most similar labelled products as evidence and chooses a leaf of the tree. When it is unsure, it flags the row for review. A reviewer's correction can become a **memo** instruction, which a person approves before it is used. A **holdout evaluation** then checks whether the instruction improves accuracy on products the case has not seen.

The labelled products are from [Open Food Facts](https://world.openfoodfacts.org), a free food product database. The [data README](data/product-classification/README.md) explains how the files were prepared and their licence.

## Setup

Install the SDK. Colab already includes many packages, so the shared dependencies are installed first and the SDK is installed without changing them.

In [ ]:
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps "docu-devs-api-client>=1.13.0"

Next, connect to the API. Store your key in Colab Secrets as `DOCUDEVS_API_KEY` (recommended), in an environment variable of the same name, or set `API_KEY` below. The knobs control the classify, evaluation and agent steps. The defaults use the MINI model tier and a small token budget. `CLASSIFY_SETTINGS` is passed to every classify job, so the same tier and budget apply to each run.

In [ ]:
import csv
import io
import json
import os
import random
import urllib.request
from importlib.metadata import version as package_version
from pathlib import Path

REQUIRED_SDK = (1, 13, 0)
installed_sdk = package_version("docu-devs-api-client")
if tuple(int(part) for part in installed_sdk.split(".")[:3]) < REQUIRED_SDK:
    raise RuntimeError(
        f"docu-devs-api-client {installed_sdk} is older than 1.13.0, which this notebook needs. "
        "Run the install cell again, then restart the runtime."
    )
print(f"docu-devs-api-client {installed_sdk}")

from docudevs import DocuDevsClient, classify_scores

try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
print("Connected to DocuDevs API")

In [ ]:
LLM_LEVEL = "MINI"
TARGET_DEPTH = [1, 2, 3]             # tree levels reported with each classify result
CLASSIFY_BUDGET_TOKENS = 600_000
CLASSIFY_SETTINGS = {
    "llm_level": LLM_LEVEL,
    "target_depth": TARGET_DEPTH,
    "max_tokens": CLASSIFY_BUDGET_TOKENS,
    "concurrency": 4,
}
HOLDOUT_EVAL_SAMPLE = 100
RUN_AGENT = False                    # True starts the optional headless investigation
DELETE_CASE_AT_END = True

## 1. Load the Data

The notebook reads three small files from `data/product-classification/` in this repository, or from GitHub when it runs in Colab:

- `tree.json` is the category tree. Each node has a dotted `code` such as `1.2.6`, the `parent` code (`null` for a root) and an English `name`.
- `records.csv` holds 2,500 labelled products. The `leaf` column is the label: the code of the product's most specific category. A label is always a leaf of the tree.
- `holdout.csv` holds 100 products with the same columns. The classifier never sees their `leaf`; it is the answer we compare against.

The EAN (`ean`) is kept as text, so codes with leading zeros are not changed.

In [ ]:
DATA_PATH = Path("data/product-classification")
RAW_URL = "https://raw.githubusercontent.com/docudevs/python-examples/main/data/product-classification"


def read_data_file(name):
    local_file = DATA_PATH / name
    if local_file.exists():
        return local_file.read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{RAW_URL}/{name}", timeout=120) as response:
        return response.read().decode("utf-8")


tree = json.loads(read_data_file("tree.json"))
records = list(csv.DictReader(io.StringIO(read_data_file("records.csv"))))
holdout = list(csv.DictReader(io.StringIO(read_data_file("holdout.csv"))))

nodes_by_code = {node["code"]: node for node in tree}
parent_codes = {node["parent"] for node in tree}
leaves = [node["code"] for node in tree if node["code"] not in parent_codes]
records_by_ean = {row["ean"]: row for row in records}
print(f"{len(tree)} categories, {len(leaves)} leaves, {len(records):,} records, {len(holdout)} holdout products")

Each record is a product with a label. Here are two records and the path of the category their label names:

In [ ]:
def category_path(code):
    path = []
    while code:
        path.append(nodes_by_code[code]["name"])
        code = nodes_by_code[code]["parent"]
    return " > ".join(reversed(path))


for row in records[:2]:
    print(f"{row['name']} ({row['brand'] or 'no brand'}, {row['quantity'] or 'no size'})")
    print(f"  label {row['leaf']}: {category_path(row['leaf'])}")
print("Root categories:", ", ".join(node["name"] for node in tree if node["parent"] is None))

## 2. Create a Case and Import the Records

`create_records_case` creates the case and configures its records dataset in one call. A records case holds only records: there are no documents and no extraction step. The arguments tell the case how to read each record:

- `key` is the record's unique id, here the EAN.
- `text` lists the text fields the classifier reads. The embedded text used for similarity search is those fields, one per line, with empty fields skipped, so no template is needed.
- `label` is the column that holds the answer, here the leaf code.
- `metadata` keeps values with the record without searching them, here the brand.
- `target_depth` sets the tree levels reported with each classify result.

In [ ]:
TEXT_FIELDS = ["name", "generic_name", "brand", "quantity", "ingredients"]

case = await client.create_records_case(
    "Product classification (notebook)",
    key="ean",
    text=TEXT_FIELDS,
    label="leaf",
    metadata=["brand"],
    target_depth=TARGET_DEPTH,
    index_settings={"top_k": 10, "hybrid": False},
)
print(f"Created case {case.id}")

Now import the 2,500 records. `import_records_and_wait` uploads the rows, waits until the import is complete and returns an `ImportJob` with the row counts. It raises an error if the import fails.

In [ ]:
imported = await case.import_records_and_wait(records)
print(
    f"Received {imported.rows_received}, rejected {imported.rows_rejected}, "
    f"added {imported.records_added}"
)

## 3. Upload and Activate the Tree

Trees are versioned. `set_tree_and_wait` uploads the node list as a draft version, activates it and waits for activation. Activation checks that every record label exists in the tree, so a tree that is missing a label is rejected. `upload_tree` and `activate_tree` are available when you want to check a draft before activating it.

In [ ]:
activation = await case.set_tree_and_wait(tree, name="Open Food Facts categories")
print(f"Tree version {activation.id} is active with {len(leaves)} leaves")

## 4. Classify the Held-Out Products

The held-out products are sent inline, without their labels. `case.classify` returns a job handle as soon as the job is accepted; the classification runs on the server. `await job.wait()` blocks until it finishes. Plain rows are converted with the case's field names, so the CSV rows are passed as they are and the `leaf` column is never sent. The job pins the tree, memo and records versions when it starts, so later changes do not affect a running job. `classify_and_wait` does both steps in one call; the review step below uses it.

Each result reports a category at every level in `TARGET_DEPTH`, and a declined product has no chosen category.

In [ ]:
job = await case.classify(holdout, **CLASSIFY_SETTINGS)
await job.wait()
print(f"Classify job {job.id}: {job.status}", job.counts)

results = await job.results()
declined = sum(1 for row in results if row.chosen is None)
print(f"{len(results)} rows, {declined} declined or failed")

Accuracy is measured per level with `classify_scores`. For a level L, a product is scored only if its label is at least L levels deep, and its answer at level L is compared with the level-L ancestor of its label, read from the tree. A declined product counts as wrong. Each level prints as correct out of scored, so a deep level can have fewer products than the shallow ones.

In [ ]:
truth_by_key = {row["ean"]: row["leaf"] for row in holdout}
for level, score in classify_scores(results, truth_by_key, tree=tree).items():
    print(f"Level {level}: {score} correct")

### Flagged rows

The classifier flags the rows it is unsure about for review. `job.flagged()` fetches only those rows. Each flagged row lists its reasons, the model's alternative when it has one, and the records it used as evidence. Here are the first few:

In [ ]:
flagged = await job.flagged()
print(f"{len(flagged)} of {len(results)} rows need review")

for row in flagged[:5]:
    print(f"\nRow {row.row_id}: {row.key} (historical label {truth_by_key[row.key]})")
    print(f"  Chosen: {' > '.join(row.chosen.path) if row.chosen else 'declined'}")
    print(f"  Reasons: {', '.join(row.review_reasons)}")
    if row.alternative:
        print(f"  Model's alternative: {row.alternative.get('reason')}")
    for ean in row.evidence_keys[:3]:
        evidence = records_by_ean.get(ean, {})
        print(f"  Evidence: {evidence.get('name', ean)} -> {evidence.get('leaf', '?')}")

## 5. Review a Sample of Records

Review works on records that are already in the case, so we classify a small batch of record keys and compare the results with the labels the records already have. The batch is drawn from the records, not from the holdout, so the review does not touch the holdout.

- `accept()` confirms a correct result and writes it as the record's review label.
- `correct(code, reason=...)` writes the category you supply and adds a correction memo entry with your reason.

In [ ]:
REVIEW_SEED = 20261013
REVIEW_BATCH = 30
review_keys = [row["ean"] for row in random.Random(REVIEW_SEED).sample(records, REVIEW_BATCH)]
review_job = await case.classify_and_wait(keys=review_keys, **CLASSIFY_SETTINGS)
review_rows = await review_job.results()


def agrees_with_label(row):
    return row.chosen.code == records_by_ean[row.key]["leaf"]


answered = [row for row in review_rows if row.chosen]
agree = [row for row in answered if agrees_with_label(row)]
disagree = [row for row in answered if not agrees_with_label(row)]
print(f"Review batch: {len(agree)} agree with the record label, {len(disagree)} disagree")

`accept()` and `correct()` each wait for the record's label write before they return, so the record can be read back straight away. The client reads the record, because the case handle does not read records. The first agreeing row is accepted:

In [ ]:
accept_row = agree[0]
await accept_row.accept()
accepted_label = (await case.get_record(accept_row.key)).label
print(f"Accepted {accept_row.key}: label is {accepted_label}")

The first disagreeing row is corrected to the record's label. The reason is stored with the correction memo entry.

In [ ]:
correct_row = disagree[0]
historical_code = records_by_ean[correct_row.key]["leaf"]
await correct_row.correct(
    historical_code,
    reason="Reviewed: the product name and ingredients describe the historical category, not the one the model chose.",
)
corrected_label = (await case.get_record(correct_row.key)).label
print(
    f"Corrected {correct_row.key}: model said {correct_row.chosen.code}, "
    f"review label is {corrected_label}"
)

## 6. Memo: Write an Instruction from the Correction

The correction shows the model choosing one category when the reviewer expects a sibling. We turn that into a memo **instruction** that covers the subtree of the parent category. `case.memo.add` proposes the entry for that subtree. The instruction starts as `proposed` and changes nothing until a person approves it, so `approve()` is the step that makes it active.

In [ ]:
truth_name = nodes_by_code[historical_code]["name"]
predicted_name = nodes_by_code[correct_row.chosen.code]["name"]
rule_text = (
    f"Between '{truth_name}' and '{predicted_name}': file a product under '{truth_name}' "
    f"when its name or generic name describes that product type. Use '{predicted_name}' only "
    "when the product name names that type."
)
parent = nodes_by_code[historical_code]["parent"] or historical_code
entry = await case.memo.add(rule_text, scope=parent)
await entry.approve()
print(f"Approved memo entry {entry.id} (status {entry.status})")
print(rule_text)

### Holdout evaluation with and without the memo

The [holdout evaluation](https://docs.docudevs.ai/advanced/case-evals) is a server-side test. It draws its own sample of labelled records (not the holdout list above) and keeps near-duplicates out of each sampled record's evidence. Each variant is a name with its overrides, and the first variant is the baseline. Both variants use that sample: `without-memo` ignores the memo, and `with-memo` uses the active memo, which now includes the instruction. `holdout_eval_and_wait` starts the evaluation and returns once its report is ready.

The rule came from a review correction, not from the evaluation sample. One caveat: the correction is a memo entry for one record, so if that record is in the sample, its own result reflects the correction.

In [ ]:
ev = await case.holdout_eval_and_wait(
    {"without-memo": {"memo": "none"}, "with-memo": {}},
    sample_size=HOLDOUT_EVAL_SAMPLE,
    seed=7,
    llm_level=LLM_LEVEL,
    target_depth=TARGET_DEPTH,
)
print(ev.report.summary())

## 7. Optional: Investigate the Flagged Rows with the Agent

A headless run works through the flagged rows of the classify job with a small token budget. `job=job` ties the run to that classify job. It cannot confirm anything: its proposals wait in the pending-action inbox for a person, which `pending_actions()` lists. Set `RUN_AGENT = True` to start it. The [chat agent](https://docs.docudevs.ai/advanced/case-agent) can also answer questions about a single row.

In [ ]:
if RUN_AGENT:
    run = await case.agent_run_and_wait(
        "investigate-flagged", job=job, params={"max_rows": 3}, max_tokens=20_000
    )
    print(run.status, "| tokens:", run.tokens_used, "| note:", run.error)
    print((run.report or {}).get("summary"))
    for action in await case.pending_actions():
        print(action.status, "-", action.summary)
else:
    print("Set RUN_AGENT = True to start a headless investigation (20k token budget).")

## 8. Cleanup

Deleting the case removes its records, results, memo and evaluations. Set `DELETE_CASE_AT_END = False` to keep the case and explore it in the web app.

In [ ]:
if DELETE_CASE_AT_END:
    await case.delete()
    print(f"Deleted case {case.id}")
else:
    print(f"Kept case {case.id}")

## Next Steps

- [Records datasets](https://docs.docudevs.ai/advanced/records-datasets): field mapping, imports and search
- [Classification](https://docs.docudevs.ai/advanced/classification): candidate strategy, flags and routing
- [Case memo](https://docs.docudevs.ai/advanced/case-memo) and [holdout evaluation](https://docs.docudevs.ai/advanced/case-evals)
- [Headless agent runs](https://docs.docudevs.ai/advanced/agent-runs)

Data: [Open Food Facts](https://world.openfoodfacts.org), ODbL 1.0 and DbCL 1.0. Contains data from Open Food Facts, available under the Open Database License. See the [data README](data/product-classification/README.md).